# Model v1 - CatBoost baseline for VIEWS forecasting


Clean, explainable baseline. Heavy diagnostics live in `notebooks/EDA.ipynb`.
Run from the repo root so relative paths resolve correctly.


## 0. Setup
Short config for paths and modeling knobs.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

BASE_DIR = Path.cwd()
ALLDATA_PATH = BASE_DIR / "data" / "AllData.csv"
TESTDATA_PATH = BASE_DIR / "data" / "TestDataset.csv"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
OUTPUTS_DIR = BASE_DIR / "outputs"
ARTIFACTS_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)

HOLDOUT_DAYS = 30
LOSS_FUNCTION = "MAE"
USE_CHANNEL_ID = False
DEDUPLICATE = True

ALPHA_CHANNEL = 10.0
ALPHA_SLOPE = 50.0
MIN_SLOPE_ROWS = 20
USE_PRED_CLIP = True
PRED_CLIP_Q = 0.65
MIN_CLIP_ROWS = 20
BLEND_ALPHA = 0.4
SCALE_FACTOR = 1.0


## 1. Load data
Minimal parsing. EDA contains the full diagnostics.


In [ ]:
df = pd.read_csv(ALLDATA_PATH)
df.columns = df.columns.str.strip()
df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce")


## 1.1 Optional dedup
Reduce label noise for identical (CPM, CHANNEL_NAME, DATE).


In [ ]:
if DEDUPLICATE:
    key = ["CHANNEL_NAME", "DATE", "CPM"]
    df = (
        df.groupby(key, as_index=False)
          .agg(
              VIEWS=("VIEWS", "median"),
              CLICKS=("CLICKS", "sum"),
              ACTIONS=("ACTIONS", "sum"),
              dup_count=("VIEWS", "size"),
          )
    )
    df["dup_count"] = df["dup_count"].astype(int)


## 2. Time-based split
Holdout on the last N dates for a simple sanity check.


In [ ]:
def split_last_days(data: pd.DataFrame, holdout_days: int = 30):
    unique_dates = np.sort(data["DATE"].unique())
    cutoff = unique_dates[-holdout_days]
    train = data.loc[data["DATE"] < cutoff].copy()
    valid = data.loc[data["DATE"] >= cutoff].copy()
    return train, valid

train_df, valid_df = split_last_days(df, holdout_days=HOLDOUT_DAYS)


## 3. Feature helpers
Keep them compact and human-readable.


In [ ]:
def add_date_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()
    d = out["DATE"]
    out["dow"] = d.dt.dayofweek.astype(int)
    out["is_weekend"] = (out["dow"] >= 5).astype(int)
    out["month"] = d.dt.month.astype(int)
    out["dayofyear"] = d.dt.dayofyear.astype(int)
    out["doy_sin"] = np.sin(2 * np.pi * out["dayofyear"] / 366.0)
    out["doy_cos"] = np.cos(2 * np.pi * out["dayofyear"] / 366.0)
    return out


def apply_basic_preprocess(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()
    out["cpm"] = out["CPM"].astype(float)
    out["log_cpm"] = np.log1p(out["cpm"].clip(lower=0))
    out = add_date_features(out)
    return out


In [ ]:
def _safe_slope(x, y):
    if len(x) < 2 or np.var(x) == 0:
        return np.nan
    return float(np.polyfit(x, y, 1)[0])


def fit_channel_stats(train: pd.DataFrame, alpha: float = 10.0):
    y = np.log1p(train["VIEWS"].clip(lower=0))
    x = np.log1p(train["CPM"].clip(lower=0))

    global_med = float(np.median(y))
    global_ctr = float(train["CLICKS"].sum() / train["VIEWS"].sum())
    global_actions_rate = float(train["ACTIONS"].sum() / train["VIEWS"].sum())
    global_slope = _safe_slope(x, y)
    global_cpm_log_med = float(np.median(x))
    global_clip = float(train["VIEWS"].quantile(PRED_CLIP_Q))

    stats = (
        train.assign(y=y, log_cpm=x)
             .groupby("CHANNEL_NAME")
             .agg(
                 ch_count=("VIEWS", "size"),
                 ch_med=("y", "median"),
                 views_sum=("VIEWS", "sum"),
                 clicks_sum=("CLICKS", "sum"),
                 actions_sum=("ACTIONS", "sum"),
                 ch_cpm_log_med=("log_cpm", "median"),
             )
             .reset_index()
    )

    stats["ch_ctr_raw"] = stats["clicks_sum"] / stats["views_sum"].replace(0, np.nan)
    stats["ch_actions_rate_raw"] = stats["actions_sum"] / stats["views_sum"].replace(0, np.nan)

    def _channel_slope(g):
        if len(g) < MIN_SLOPE_ROWS or g["CPM"].nunique() < 2:
            return np.nan
        gx = np.log1p(g["CPM"].clip(lower=0))
        gy = np.log1p(g["VIEWS"].clip(lower=0))
        return _safe_slope(gx, gy)

    ch_slope = (
        train.groupby("CHANNEL_NAME")
             .apply(_channel_slope)
             .rename("ch_slope_raw")
             .reset_index()
    )

    stats = stats.merge(ch_slope, on="CHANNEL_NAME", how="left")

    w = stats["ch_count"] / (stats["ch_count"] + alpha)
    stats["ch_med_smooth"] = w * stats["ch_med"] + (1 - w) * global_med
    stats["ch_ctr_smooth"] = w * stats["ch_ctr_raw"].fillna(global_ctr) + (1 - w) * global_ctr
    stats["ch_actions_rate_smooth"] = (
        w * stats["ch_actions_rate_raw"].fillna(global_actions_rate)
        + (1 - w) * global_actions_rate
    )

    w_slope = stats["ch_count"] / (stats["ch_count"] + ALPHA_SLOPE)
    stats["ch_slope_smooth"] = (
        w_slope * stats["ch_slope_raw"].fillna(global_slope)
        + (1 - w_slope) * global_slope
    )

    ch_clip = (
        train.groupby("CHANNEL_NAME")["VIEWS"]
             .quantile(PRED_CLIP_Q)
             .rename("ch_views_clip")
             .reset_index()
    )
    stats = stats.merge(ch_clip, on="CHANNEL_NAME", how="left")
    stats.loc[stats["ch_count"] < MIN_CLIP_ROWS, "ch_views_clip"] = np.nan

    keep_cols = [
        "CHANNEL_NAME",
        "ch_count",
        "ch_med_smooth",
        "ch_ctr_smooth",
        "ch_actions_rate_smooth",
        "ch_cpm_log_med",
        "ch_slope_smooth",
        "ch_views_clip",
    ]

    return (
        stats[keep_cols],
        global_med,
        global_ctr,
        global_actions_rate,
        global_slope,
        global_cpm_log_med,
        global_clip,
    )


def apply_channel_stats(
    data: pd.DataFrame,
    ch_stats: pd.DataFrame,
    global_med: float,
    global_ctr: float,
    global_actions_rate: float,
    global_slope: float,
    global_cpm_log_med: float,
    global_clip: float,
) -> pd.DataFrame:
    out = data.merge(ch_stats, on="CHANNEL_NAME", how="left")
    out["ch_count"] = out["ch_count"].fillna(0).astype(int)
    out["ch_count_log"] = np.log1p(out["ch_count"])
    out["ch_med_smooth"] = out["ch_med_smooth"].fillna(global_med).astype(float)
    out["ch_ctr_smooth"] = out["ch_ctr_smooth"].fillna(global_ctr).astype(float)
    out["ch_actions_rate_smooth"] = out["ch_actions_rate_smooth"].fillna(global_actions_rate).astype(float)
    out["ch_slope_smooth"] = out["ch_slope_smooth"].fillna(global_slope).astype(float)
    out["ch_cpm_log_med"] = out["ch_cpm_log_med"].fillna(global_cpm_log_med).astype(float)
    out["ch_views_clip"] = out["ch_views_clip"].fillna(global_clip).astype(float)

    out["ch_log_pred"] = out["ch_med_smooth"] + out["ch_slope_smooth"] * (
        out["log_cpm"] - out["ch_cpm_log_med"]
    )
    return out


In [ ]:
def fit_channel_cpm_stats(train: pd.DataFrame):
    stats = (
        train.groupby("CHANNEL_NAME")["CPM"]
             .median()
             .reset_index(name="ch_cpm_median")
    )
    global_median = float(train["CPM"].median())
    return stats, global_median


def apply_channel_cpm_stats(data: pd.DataFrame, ch_stats: pd.DataFrame, global_median: float) -> pd.DataFrame:
    out = data.merge(ch_stats, on="CHANNEL_NAME", how="left")
    out["ch_cpm_median"] = out["ch_cpm_median"].fillna(global_median)
    denom = out["ch_cpm_median"].replace(0, np.nan)
    out["cpm_to_ch_median"] = (out["cpm"] / denom).fillna(1.0)
    return out


In [ ]:
def post_process_predictions(pred, feats):
    out = np.clip(pred, 0, None)
    if SCALE_FACTOR != 1.0:
        out = out * SCALE_FACTOR
    if BLEND_ALPHA >= 0:
        base = np.expm1(feats["ch_med_smooth"].to_numpy())
        out = BLEND_ALPHA * out + (1 - BLEND_ALPHA) * base
    if USE_PRED_CLIP:
        out = np.minimum(out, feats["ch_views_clip"].to_numpy())
    return out


## 4. Build matrices


In [ ]:
train_fe = apply_basic_preprocess(train_df)
valid_fe = apply_basic_preprocess(valid_df)

(
    ch_stats,
    global_med_log,
    global_ctr,
    global_actions_rate,
    global_slope,
    global_cpm_log_med,
    global_clip,
) = fit_channel_stats(train_df, alpha=ALPHA_CHANNEL)

train_fe = apply_channel_stats(
    train_fe,
    ch_stats,
    global_med_log,
    global_ctr,
    global_actions_rate,
    global_slope,
    global_cpm_log_med,
    global_clip,
)
valid_fe = apply_channel_stats(
    valid_fe,
    ch_stats,
    global_med_log,
    global_ctr,
    global_actions_rate,
    global_slope,
    global_cpm_log_med,
    global_clip,
)

ch_cpm_stats, global_cpm_median = fit_channel_cpm_stats(train_df)
train_fe = apply_channel_cpm_stats(train_fe, ch_cpm_stats, global_cpm_median)
valid_fe = apply_channel_cpm_stats(valid_fe, ch_cpm_stats, global_cpm_median)

feature_cols = [
    "log_cpm",
    "cpm_to_ch_median",
    "dow", "is_weekend", "month", "doy_sin", "doy_cos",
    "ch_count_log", "ch_med_smooth", "ch_ctr_smooth", "ch_actions_rate_smooth",
    "ch_slope_smooth", "ch_log_pred",
]

cat_features = []
if USE_CHANNEL_ID:
    feature_cols.append("CHANNEL_NAME")
    cat_features = ["CHANNEL_NAME"]

X_train = train_fe[feature_cols]
X_valid = valid_fe[feature_cols]

y_train_raw = train_df["VIEWS"].astype(float).values
y_valid_raw = valid_df["VIEWS"].astype(float).values

y_train = np.log1p(np.clip(y_train_raw, 0, None))
y_valid = np.log1p(np.clip(y_valid_raw, 0, None))

train_weights = train_df["dup_count"].values if "dup_count" in train_df.columns else None
valid_weights = valid_df["dup_count"].values if "dup_count" in valid_df.columns else None

train_pool = Pool(X_train, y_train, cat_features=cat_features, weight=train_weights)
valid_pool = Pool(X_valid, y_valid, cat_features=cat_features, weight=valid_weights)


## 5. Train model


In [ ]:
model = CatBoostRegressor(
    loss_function=LOSS_FUNCTION,
    depth=8,
    learning_rate=0.05,
    iterations=4000,
    l2_leaf_reg=5,
    random_strength=1.0,
    bootstrap_type="Bayesian",
    bagging_temperature=0.8,
    random_seed=RANDOM_SEED,
    eval_metric=LOSS_FUNCTION,
    verbose=500,
    od_type="Iter",
    od_wait=200,
    task_type="CPU",
    devices="0",
)

model.fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=500)


## 6. Validation


In [ ]:
def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def rmsle(y_true, y_pred):
    y_true = np.clip(np.asarray(y_true, dtype=float), 0, None)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2)))

def smape(y_true, y_pred, eps=1e-8):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.abs(y_true) + np.abs(y_pred) + eps
    return float(np.mean(2.0 * np.abs(y_pred - y_true) / denom))

pred_valid_log = model.predict(valid_pool)
pred_valid_raw = np.clip(np.expm1(pred_valid_log), 0, None)
pred_valid = post_process_predictions(pred_valid_raw, valid_fe)

metrics = {
    "MAE": mae(y_valid_raw, pred_valid),
    "RMSE": rmse(y_valid_raw, pred_valid),
    "RMSLE": rmsle(y_valid_raw, pred_valid),
    "SMAPE": smape(y_valid_raw, pred_valid),
}
metrics


## 7. Train on full data and save artifacts


In [ ]:
full_df = df.copy()
full_fe = apply_basic_preprocess(full_df)

(
    ch_stats_full,
    global_med_log_full,
    global_ctr_full,
    global_actions_rate_full,
    global_slope_full,
    global_cpm_log_med_full,
    global_clip_full,
) = fit_channel_stats(full_df, alpha=ALPHA_CHANNEL)

full_fe = apply_channel_stats(
    full_fe,
    ch_stats_full,
    global_med_log_full,
    global_ctr_full,
    global_actions_rate_full,
    global_slope_full,
    global_cpm_log_med_full,
    global_clip_full,
)

ch_cpm_stats_full, global_cpm_median_full = fit_channel_cpm_stats(full_df)
full_fe = apply_channel_cpm_stats(full_fe, ch_cpm_stats_full, global_cpm_median_full)

X_full = full_fe[feature_cols]
y_full_raw = full_df["VIEWS"].astype(float).values
y_full_log = np.log1p(np.clip(y_full_raw, 0, None))

full_weights = full_df["dup_count"].values if "dup_count" in full_df.columns else None
full_pool = Pool(X_full, y_full_log, cat_features=cat_features, weight=full_weights)

final_model = CatBoostRegressor(**model.get_params())
final_model.fit(full_pool, verbose=200)

final_model.save_model(ARTIFACTS_DIR / "model_v1.cbm")
ch_stats_full.to_csv(ARTIFACTS_DIR / "channel_stats_v1.csv", index=False)
ch_cpm_stats_full.to_csv(ARTIFACTS_DIR / "channel_cpm_stats_v1.csv", index=False)

meta = {
    "model": "CatBoostRegressor",
    "target": "log1p(VIEWS)",
    "feature_cols": feature_cols,
    "cat_features": cat_features,
    "holdout_days": HOLDOUT_DAYS,
    "loss_function": LOSS_FUNCTION,
    "notes": "Model v1 baseline",
}
(ARTIFACTS_DIR / "meta_v1.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
(ARTIFACTS_DIR / "metrics_v1.json").write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8")


## 8. Create submission


In [ ]:
test_df = pd.read_csv(TESTDATA_PATH)
test_df.columns = test_df.columns.str.strip()
test_df["DATE"] = pd.to_datetime(test_df["DATE"], errors="coerce")

test_fe = apply_basic_preprocess(test_df)
test_fe = apply_channel_stats(
    test_fe,
    ch_stats_full,
    global_med_log_full,
    global_ctr_full,
    global_actions_rate_full,
    global_slope_full,
    global_cpm_log_med_full,
    global_clip_full,
)
test_fe = apply_channel_cpm_stats(test_fe, ch_cpm_stats_full, global_cpm_median_full)

X_test = test_fe[feature_cols]
test_pool = Pool(X_test, cat_features=cat_features)

pred_test_log = final_model.predict(test_pool)
pred_test_raw = np.clip(np.expm1(pred_test_log), 0, None)
pred_test = post_process_predictions(pred_test_raw, test_fe)

out = test_df.copy()
out["VIEWS"] = np.round(pred_test).astype(int)
out.to_csv(OUTPUTS_DIR / "TestDataset_filled_model_v1.csv", index=False)


## Notes and limits

- This baseline is explainable and stable, not aggressive.
- The public score is affected by distribution shift and label noise.
- Large gains likely require external channel metadata.
